# SAÉ S2 : LABYRINTHES
**BUT Informatique de Reims - Saison 2026**

Binôme :
- Étudiant 1 : `<Arthur BRODA (brod0022)>`
- Étudiant 2 : `<Ethan PILIA (pili0002)>`

Dépôt GitLab : `<https://iut-info.univ-reims.fr/gitlab/brod0022/sae-s2-maze.git>`

---
> Ce notebook contient les **prototypes** de toutes les fonctions à implémenter.  
> Complétez chaque cellule marquée `# TODO` sans modifier les signatures.
> 
> Complétez le docstring en indiquant quel membre du binôme a compris et co-implémenté (**en retirant votre login, vous ne pourrez pas être interrogé à l'oral sur la fonction concernée**)
> 
> Effectuez des commits réguliers (au moins un par fonction).


## Imports

In [1]:
import random
import heapq
import time


---
## Classe `Maze`

La classe principale du projet. Le constructeur, `__str__`, `info`, `fill`, `empty`, `display` / `overlay` **vous sont fournis**.

Les méthodes marquées `😣 À faire` sont celles que vous devez implémenter.


In [2]:
class Maze:
    """
    Classe Labyrinthe
    Représentation sous forme de graphe non-orienté
    dont chaque sommet est une cellule (un tuple (l,c)) d'une grille,
    et dont la structure est représentée par un dictionnaire
      - clés : sommets
      - valeurs : ensemble des sommets voisins accessibles
    Attributs :
        height (int): hauteur de la grille du labyrinthe
        width  (int): largeur de la grille du labyrinthe
        neighbors (dict): dictionnaire des voisinages
        weight    (dict): dictionnaire des poids des déplacements (labyrinthe valué)
    """

    # ──────────────────────────────────────────────────────────────────────────
    # Constructeur et méthodes utilitaires (FOURNIES - NE PAS MODIFIER)
    # ──────────────────────────────────────────────────────────────────────────

    def __init__(self, height, width, empty=False):
        """
        Constructeur de labyrinthe à 'height' lignes et 'width' colonnes.
        Arguments:
            height (int) : nombre de lignes
            width  (int) : nombre de colonnes
            empty  (bool): True → labyrinthe sans murs ; False → tous les murs
        """
        self.weight = dict()
        self.height = height
        self.width  = width
        self.neighbors = dict()
        if empty:
            for i in range(self.height):
                for j in range(self.width):
                    self.neighbors[(i, j)] = set(
                        (ii, jj)
                        for ii, jj in [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]
                        if 0 <= ii < self.height and 0 <= jj < self.width
                    )
        else:
            self.neighbors = {(i, j): set()
                              for i in range(height) for j in range(width)}

    def info(self):
        """Affichage des attributs (utile pour déboguer)."""
        txt  = "**Informations sur le labyrinthe**\n"
        txt += f"- Dimensions de la grille : {self.height} x {self.width}\n"
        txt += "- Voisinages :\n" + str(self.neighbors) + "\n"
        valid = True
        for c1 in {(i,j) for i in range(self.height) for j in range(self.width)}:
            for c2 in self.neighbors[c1]:
                if c1 not in self.neighbors[c2]:
                    valid = False
                    break
            else:
                continue
            break
        txt += "- Structure cohérente\n" if valid else f"- Structure incohérente : {c1} X {c2}\n"
        return txt

    def __str__(self):
        """Représentation ASCII du labyrinthe."""
        txt = "┏"
        for j in range(self.width-1):
            txt += "━━━┳"
        txt += "━━━┓\n┃"
        for j in range(self.width-1):
            txt += "   ┃" if (0,j+1) not in self.neighbors[(0,j)] else "    "
        txt += "   ┃\n"
        for i in range(self.height-1):
            txt += "┣"
            for j in range(self.width-1):
                txt += "━━━╋" if (i+1,j) not in self.neighbors[(i,j)] else "   ╋"
            txt += "━━━┫\n" if (i+1,self.width-1) not in self.neighbors[(i,self.width-1)] else "   ┫\n"
            txt += "┃"
            for j in range(self.width):
                txt += "   ┃" if (i+1,j+1) not in self.neighbors[(i+1,j)] else "    "
            txt += "\n"
        txt += "┗"
        for i in range(self.width-1):
            txt += "━━━┻"
        txt += "━━━┛\n"
        return txt

    def fill(self):
        """Remplit le labyrinthe avec tous les murs possibles."""
        for i in range(self.height):
            for j in range(self.width):
                if j+1 < self.width:  self.add_wall((i,j),(i,j+1))
                if i+1 < self.height: self.add_wall((i,j),(i+1,j))

    def empty(self):
        """Vide le labyrinthe de tous ses murs."""
        for i in range(self.height):
            for j in range(self.width):
                if j+1 < self.width:  self.remove_wall((i,j),(i,j+1))
                if i+1 < self.height: self.remove_wall((i,j),(i+1,j))

    def overlay(self, content=None):
        """Rendu ASCII avec contenu dans les cellules."""
        if content is None:
            content = {(i,j):' ' for i in range(self.height) for j in range(self.width)}
        else:
            new_content = {(i,j):' ' for i in range(self.height)
                           for j in range(self.width) if (i,j) not in content}
            content = {**content, **new_content}
        txt = "┏"
        for j in range(self.width-1): txt += "━━━┳"
        txt += "━━━┓\n┃"
        for j in range(self.width-1):
            txt += " "+content[(0,j)]+" ┃" if (0,j+1) not in self.neighbors[(0,j)] else " "+content[(0,j)]+"  "
        txt += " "+content[(0,self.width-1)]+" ┃\n"
        for i in range(self.height-1):
            txt += "┣"
            for j in range(self.width-1):
                txt += "━━━╋" if (i+1,j) not in self.neighbors[(i,j)] else "   ╋"
            txt += "━━━┫\n" if (i+1,self.width-1) not in self.neighbors[(i,self.width-1)] else "   ┫\n"
            txt += "┃"
            for j in range(self.width):
                txt += " "+content[(i+1,j)]+" ┃" if (i+1,j+1) not in self.neighbors[(i+1,j)] else " "+content[(i+1,j)]+"  "
            txt += "\n"
        txt += "┗"
        for i in range(self.width-1): txt += "━━━┻"
        txt += "━━━┛\n"
        return txt

    def display(self, content=None):
        """Alias de overlay."""
        return self.overlay(content)

    # ──────────────────────────────────────────────────────────────────────────
    # 1. Manipulation des murs  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def add_wall(self, c1, c2):
        """
        Ajout d'un mur entre deux cellules.
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Membre(s) du binôme concerné(s) : A fait : brod0022 / A compris : pili0002
        """
        self.neighbors[c1].remove(c2)
        self.neighbors[c2].remove(c1)

    def remove_wall(self, c1, c2):
        """
        Suppression du mur entre deux cellules.
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Membre(s) du binôme concerné(s) : A fait : brod0022 / A compris : pili0002
        """
        self.neighbors[c1].add(c2)
        self.neighbors[c2].add(c1)


    def get_all_walls(self):
        """
        Retourne la liste de tous les murs du labyrinthe
        sous la forme d'une liste de paires de cellules.
        Retour:
            list : liste de [c1, c2] représentant chaque mur
        Membre(s) du binôme concerné(s) : A fait : brod0022 / A compris : pili0002
        """
        lst = []
        for c1 in self.neighbors:
            c2 = (c1[0]+1,c1[1])
            if c2 in self.neighbors and c2 not in self.neighbors[c1]:
                lst.append((c1,c2))
            c3 = (c1[0],c1[1]+1)
            if c3 in self.neighbors and c3 not in self.neighbors[c1]:
                lst.append((c1,c3))
        return lst
        
    def get_reachable_cells(self, c):
        """
        Liste des cellules accessibles depuis c
        (cellules contigües sans mur entre elles et c).
        Argument:
            c (tuple) : cellule de référence
        Retour:
            list : cellules accessibles
        Membre(s) du binôme concerné(s) : A fait : pili0002
        """
        accessibles = []
        for voisine in self.get_contiguous_cells(c):
            if (c, voisine) in self.neighbors and (voisine, c) in self.neighbors :
                accessibles.append(voisine)
        return accessibles

    def get_contiguous_cells(self, c):
        """
        Liste des cellules contigues à c
        (cellules adjacentes sur la grille indépendamment des murs ou de l'absence de mur).
        Argument:
            c (tuple) : cellule de référence
        Retour:
            list : cellules accessibles
        Membre(s) du binôme concerné(s) : brod0022
        """
        i, j = c
        result = []
        candidates = [(i-1, j), (i+1, j), (i, j-1), (i, j+1)]

        for x, y in candidates:
            if 0 <= x < self.height and 0 <= y < self.width:
                result.append((x, y))

        return result


    # ──────────────────────────────────────────────────────────────────────────
    # 2. Génération  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    @classmethod
    def gen_btree(cls, h, w):
        """
        Génération par la méthode de l'arbre binaire.
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : brod0022
        """
        laby = cls(h, w)

        for i in range(h):
            for j in range(w):
                if not (i == 0 and j == 0):
                    if i == 0:
                        laby.remove_wall((i, j), (i, j-1))
                    elif j == 0:
                        laby.remove_wall((i, j), (i-1, j))
                    else:
                        if random.choice([True, False]):
                            laby.remove_wall((i, j), (i, j-1))
                        else:
                            laby.remove_wall((i, j), (i-1, j))

        return laby
        

    @classmethod
    def gen_sidewinder(cls, h, w):
        """
        Génération par l'algorithme sidewinder.
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : brod0022
        """
        laby = cls(h, w)

        for ligne in range(h):
            sequence = []

            for colonne in range(w):
                cellule = (ligne, colonne)
                sequence.append(cellule)

                bord_est = (colonne == w - 1)
                bord_sud = (ligne == h - 1)

                if bord_est and bord_sud:
                    pass
                elif bord_est:
                    laby.remove_wall(cellule, (ligne + 1, colonne))
                elif bord_sud:
                    laby.remove_wall(cellule, (ligne, colonne + 1))
                else:
                    if random.choice([True, False]):
                        laby.remove_wall(cellule, (ligne, colonne + 1))
                    else:
                        cellule_choisie = random.choice(sequence)
                        voisin = (cellule_choisie[0] + 1, cellule_choisie[1])
                        laby.remove_wall(cellule_choisie, voisin)
                        sequence = []

        return laby

    @classmethod
    def gen_fusion(cls, h, w):
        """
        Génération par fusion de chemins (variante de Kruskal).
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : brod0022
        """
        laby=cls(h,w)

        label={}
        num=1
        for elt in laby.neighbors:
            label[elt]=num
            num+=1

        l_mur=laby.get_all_walls()
        random.shuffle(l_mur)

        for mur in l_mur:
            if label[mur[0]]!=label[mur[1]]:
                laby.remove_wall(mur[0],mur[1])

                lab=label[mur[1]] #retiens le label de la cellule 2
                for c in label:
                    if label[c]==lab:
                        label[c]=label[mur[0]]

        return laby

    @classmethod
    def gen_exploration(cls, h, w):
        """
        Génération par exploration exhaustive (DFS aléatoire).
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : pili0002
        """
        laby = cls(h,w)
        debut = (random.randint(0, h-1), random.randint(0, w-1))
        visite = set()
        pile = []
        visite.add(debut)
        pile.append(debut)

        while pile :
            actuel = pile.pop()

            voisins = []
            for v in laby.get_contiguous_cells(actuel):
                if v not in visite:
                    voisins.append(v)

            if voisins:
                pile.append(actuel)
                next_cell = random.choice(voisins)
                laby.remove_wall(actuel, next_cell)
                visite.add(next_cell)
                pile.append(next_cell)

        return laby

    @classmethod
    def gen_wilson(cls,h, w):
        """
        Génération par l'algorithme de Wilson (marche aléatoire sans boucle).
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : brod0022
        """
    
        laby = cls(h, w)  #tous les murs

        cells = [(i, j) for i in range(h) for j in range(w)]

        #cellule initiale
        start = random.choice(cells)
        in_maze = {start}

        while len(in_maze) < len(cells):

            #départ aléatoire hors labyrinthe
            cell = random.choice([c for c in cells if c not in in_maze])

            path = [cell]
            visited = {cell: 0}

            #marche aléatoire avec suppression de boucles
            while cell not in in_maze:
                next_cell = random.choice(laby.get_contiguous_cells(cell))

                if next_cell in visited:
                    # couper la boucle
                    idx = visited[next_cell]
                    path = path[:idx+1]
                    visited = {c: i for i, c in enumerate(path)}
                else:
                    path.append(next_cell)
                    visited[next_cell] = len(path) - 1

                cell = next_cell

            #ajout du chemin au labyrinthe
            for i in range(len(path) - 1):
                c1, c2 = path[i], path[i+1]
                laby.remove_wall(c1, c2)
                in_maze.add(c1)
                in_maze.add(c2)

        return laby

    # ──────────────────────────────────────────────────────────────────────────
    # 3. Résolution  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def solve_dfs(self, start, stop):
        """
        Résolution par parcours en profondeur (DFS).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de start à stop (liste de cellules)
        Membre(s) du binôme concerné(s) : pili0002
        """
        pile = [start]
        visite = set()
        pred = dict()
        visite.add(start)
        pred[start] = start

        while pile and stop not in visite :
            actuel = pile.pop()

            for voisin in self.neighbors[actuel]:
                if voisin not in visite:
                    visite.add(voisin)
                    pred[voisin] = actuel
                    pile.append(voisin)

        chemin = []
        c = stop

        while c != start :
            chemin.append(c)
            c = pred[c]
        chemin.append(start)
        chemin.reverse()

        return chemin
    def solve_bfs(self, start, stop):
        """
        Résolution par parcours en largeur (BFS).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de start à stop (liste de cellules)
        Membre(s) du binôme concerné(s) : pili0002
        """
        file = [start]
        visited = set()
        pred = dict()
        visited.add(start)
        pred[start] = start

        while file and stop not in visited:
            actuel = file.pop(0)

            for voisin in self.neighbors[actuel]:
                if voisin not in visited:
                    visited.add(voisin)
                    pred[voisin] = actuel
                    file.append(voisin)

        chemin = []
        c = stop
        while c!= start:
            chemin.append(c)
            c = pred[c]
        chemin.append(start)
        chemin.reverse()

        return chemin

    def solve_rhr(self, start, stop):
        """
        Résolution par la règle de la main droite (right-hand rule).
        L'algorithme longe les murs en gardant toujours un contact
        avec le mur de droite.
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin trouvé (peut contenir des cycles)
        Membre(s) du binôme concerné(s) : login1 login2
        """
        # TODO
        pass

    # ──────────────────────────────────────────────────────────────────────────
    # 4. Métriques  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────


    def get_tree(self, start):
        """
        Extraction de l'arborescence de racine 'start' à l'aide d'un parcours.
        Arguments:
            start : cellule de départ
        Retour:
            dict : dictionnaire dont les clés sont les cellules du labyrinthe 
                   et les valeurs sont les prédécesseurs respectifs de ces cellules
        Membre(s) du binôme concerné(s) : pili0002
        """
        pile = [start]
        visite = set()
        pred = dict()
        visite.add(start)
        pred[start] = start

        while pile :
            actuel = pile.pop()
            for voisin in self.neighbors[actuel]:
                if voisin not in visite:
                    visite.add(voisin)
                    pred[voisin] = actuel
                    pile.append(voisin)

        return pred



    def distance_geo(self, c1, c2):
        """
        Distance géodésique entre c1 et c2 :
        nombre minimal de déplacements sur le graphe.
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Retour:
            int : longueur du chemin minimal
        Membre(s) du binôme concerné(s) : pili0002
        """
        chemin = self.solve_bfs(c1,c2)

        if chemin == []:
            return -1

        return len(chemin) -1

    def distance_man(self, c1, c2):
        """
        Distance de Manhattan entre c1 et c2 sur la grille
        (sans tenir compte des murs).
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Retour:
            int : distance de Manhattan
        Membre(s) du binôme concerné(s) : pili0002
        """
        return abs(c1[0] - c2[0]) + abs(c1[1] - c2[1])

    def tortuosity(self, start, stop):
        """
        Rapport entre la distance géodésique et la distance de Manhattan.
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            float : tortuosité (>= 1.0)
        Membre(s) du binôme concerné(s) : pili0002
        """
        d_geo = self.distance_geo(start, stop)
        d_man = self.distance_man(start, stop)

        if d_man == 0:
            return 0

        return d_geo / d_man

    def worst_path_len(self, start, stop):
        """
        Longueur du plus long chemin menant du départ à une impasse (feuille).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            int : longueur maximale d'une branche depuis start
        Membre(s) du binôme concerné(s) : pili0002
        """
        pred = self.get_tree(start)
        max_len = 0

        for cellule in pred :
            longueur = 0
            c = cellule
            while c != start :
                c = pred[c]
                longueur += 1

            enfants = 0
            for voisin in self.neighbors[cellule]:
                if voisin in pred and pred[voisin] == cellule:
                    enfants += 1
            if enfants == 0:
                if longueur > max_len:
                    max_len = longueur

        return max_len

    def dead_end_number(self):
        """
        Nombre de culs-de-sac (cellules de degré 1).
        Retour:
            int : nombre de culs-de-sac
        Membre(s) du binôme concerné(s) : pili0002
        """
        # TODO
        pass

    def dead_end_rate(self):
        """
        Proportion de culs-de-sac par rapport au nombre total de cellules.
        Retour:
            float : taux de culs-de-sac
        Membre(s) du binôme concerné(s) : brod0022
        """
        total = self.height * self.width
        culs_de_sac = 0

        for cell in self.neighbors:
            if len(self.neighbors[cell]) == 1:
                culs_de_sac += 1

        return culs_de_sac / total

    def linearity_index(self):
        """
        Proportion de cellules de degré 2 (couloirs droits).
        Retour:
            float : index de linéarité
        Membre(s) du binôme concerné(s) : brod0022
        """
        total = self.height * self.width
        couloirs = 0

        for cell in self.neighbors:
            if len(self.neighbors[cell]) == 2:
                couloirs += 1

        return couloirs / total

    # ──────────────────────────────────────────────────────────────────────────
    # 5. Labyrinthes imparfaits et valués  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def remove_random_wall(self):
        """
        Supprime un mur au hasard dans le labyrinthe
        (crée un passage entre deux cellules adjacentes non encore reliées).
        Membre(s) du binôme concerné(s) : pili0002
        """
        while True:
            i = random.randint(0, self.height - 1)
            j = random.randint(0, self.width - 1)
            c1 = (i, j)
            voisins = self.get_contiguous_cells(c1)

            if voisins :
                c2 = random.choice(voisins)
                if c2 not in self.neighbors[c1]:
                    self.remove_wall(c1, c2)


    def add_random_weights(self, max_weight):
        """
        Ajoute un poids aléatoire (entre 1 et max_weight) à chaque
        déplacement possible. Les poids sont orientés : c1→c2 peut
        différer de c2→c1.
        Argument:
            max_weight (int) : poids maximal possible
        Membre(s) du binôme concerné(s) : brod0022
        """
        self.weight = dict()

        for c1 in self.neighbors:
            for c2 in self.neighbors[c1]:
                self.weight[(c1, c2)] = random.randint(1, max_weight)


    # ──────────────────────────────────────────────────────────────────────────
    # 6. Résolution de labyrinthes valués  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def solve_dijkstra(self, start, stop):
        """
        Chemin de coût minimal - Dijkstra version naïve (sans tas).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de coût minimal de start à stop
        Membre(s) du binôme concerné(s) : pili0002
        """
        Dist = dict()
        pred = dict()
        marquage = dict()
        for cell in self.neighbors:
            Dist[cell] = float("inf")
            pred[cell] = None
            marquage[cell] = False

        Dist[start] = 0
        pred[start] = start

        while False in marquage.values():
            mini = float("inf")
            i = cell
            for cell in self.neighbors:
                if not marquage[cell] and Dist[cell] < mini :
                    mini = Dist[cell]
                    i = cell
            marquage[i] = True
            for j in self.neighbors[i]:
                if not marquage[j]:
                    if Dist[i] + self.weight[(i, j)] < Dist[j]:
                        Dist[j] = Dist[i] + self.weight[(i, j)]
                        pred[j] = i

        chemin = []
        c = stop
        while c != start:
            chemin.append(c)
            c = pred[c]
        chemin.append(start)
        chemin.reverse()

        return chemin

    def solve_dijkstra_whq(self, start, stop):
        """
        [BONUS]
        Chemin de coût minimal - Dijkstra avec liste à priorité (heapq).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de coût minimal de start à stop
        Membre(s) du binôme concerné(s) : login1 login2
        """
        # TODO
        pass


---
## Section 1 - Manipulation des murs

### Tests de `remove_wall`


In [3]:
laby = Maze(5, 5)
print("Labyrinthe plein :")
print(laby)

laby.remove_wall((0, 0), (0, 1))
print("Après remove_wall((0,0),(0,1)) :")
print(laby)


Labyrinthe plein :
┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛

Après remove_wall((0,0),(0,1)) :
┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃       ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛



### Tests de `get_all_walls`

In [4]:
laby2 = Maze(3, 3, empty = True)
laby2.add_wall((0, 0), (0, 1))
laby2.add_wall((1, 1), (1, 2))
print("Murs du labyrinthe :")
print(laby2.get_all_walls())
print(laby2)


Murs du labyrinthe :
[((0, 0), (0, 1)), ((1, 1), (1, 2))]
┏━━━┳━━━┳━━━┓
┃   ┃       ┃
┣   ╋   ╋   ┫
┃       ┃   ┃
┣   ╋   ╋   ┫
┃           ┃
┗━━━┻━━━┻━━━┛



### Tests de `get_reachable_cells`

In [5]:
laby3 = Maze(4, 4)
laby3.neighbors = {
    (0,0): {(1,0)},
    (0,1): {(0,2),(1,1)},
    (0,2): {(0,1),(0,3)},
    (0,3): {(0,2),(1,3)},
    (1,0): {(2,0),(0,0)},
    (1,1): {(0,1),(1,2)},
    (1,2): {(1,1),(2,2)},
    (1,3): {(2,3),(0,3)},
    (2,0): {(1,0),(2,1),(3,0)},
    (2,1): {(2,0),(2,2)},
    (2,2): {(1,2),(2,1)},
    (2,3): {(3,3),(1,3)},
    (3,0): {(3,1),(2,0)},
    (3,1): {(3,2),(3,0)},
    (3,2): {(3,1)},
    (3,3): {(2,3)}
}
print("Cellules contigues de (2,0) :", laby3.get_contiguous_cells((2,0)))
print("Cellules accessibles de (2,0) :", laby3.get_reachable_cells((2,0)))


Cellules contigues de (2,0) : [(1, 0), (3, 0), (2, 1)]
Cellules accessibles de (2,0) : []


---
## Section 2 - Génération

### `gen_btree`


In [6]:
laby_btree = Maze.gen_btree(6, 6)
print(laby_btree)


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃                       ┃
┣   ╋   ╋   ╋━━━╋   ╋   ┫
┃   ┃   ┃       ┃   ┃   ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ┫
┃   ┃               ┃   ┃
┣   ╋   ╋   ╋   ╋━━━╋━━━┫
┃   ┃   ┃   ┃           ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ┫
┃   ┃               ┃   ┃
┣   ╋   ╋━━━╋   ╋   ╋   ┫
┃   ┃       ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_sidewinder`

In [7]:
laby_sw = Maze.gen_sidewinder(6, 6)
print(laby_sw)


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃                   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ┫
┃   ┃   ┃   ┃   ┃       ┃
┣   ╋   ╋   ╋   ╋━━━╋   ┫
┃   ┃       ┃   ┃   ┃   ┃
┣   ╋   ╋━━━╋   ╋   ╋   ┫
┃           ┃   ┃   ┃   ┃
┣━━━╋   ╋━━━╋   ╋   ╋   ┫
┃   ┃       ┃   ┃       ┃
┣   ╋━━━╋   ╋   ╋━━━╋   ┫
┃                       ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_fusion`

In [8]:
laby_fusion = Maze.gen_fusion(10, 10)
print(laby_fusion)


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃       ┃           ┃           ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━┫
┃   ┃               ┃   ┃               ┃
┣   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋   ╋   ┫
┃           ┃   ┃       ┃   ┃       ┃   ┃
┣━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ┫
┃       ┃       ┃       ┃           ┃   ┃
┣   ╋   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ┫
┃   ┃       ┃   ┃               ┃   ┃   ┃
┣   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ╋━━━┫
┃   ┃           ┃   ┃               ┃   ┃
┣━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ┫
┃           ┃   ┃   ┃               ┃   ┃
┣━━━╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ┫
┃                               ┃   ┃   ┃
┣━━━╋   ╋━━━╋   ╋   ╋━━━╋━━━╋━━━╋   ╋   ┫
┃           ┃   ┃                       ┃
┣━━━╋   ╋━━━╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋   ┫
┃       ┃       ┃           ┃           ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_exploration`

In [9]:
laby_exp = Maze.gen_exploration(10, 10)
print(laby_exp)


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃       ┃                           ┃   ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ┫
┃           ┃       ┃       ┃   ┃       ┃
┣   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋   ┫
┃       ┃       ┃       ┃   ┃       ┃   ┃
┣━━━╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ┫
┃       ┃   ┃       ┃           ┃   ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋   ┫
┃           ┃   ┃       ┃   ┃       ┃   ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ┫
┃       ┃           ┃       ┃   ┃       ┃
┣   ╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━┫
┃   ┃               ┃   ┃   ┃           ┃
┣   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃       ┃       ┃       ┃           ┃   ┃
┣━━━╋   ╋   ╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ┫
┃       ┃   ┃   ┃           ┃       ┃   ┃
┣   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋   ╋   ╋   ┫
┃   ┃       ┃       ┃           ┃       ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_wilson`

In [22]:
laby_wilson = Maze.gen_wilson(10,10)
print(laby_wilson)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃               ┃                       ┃
┣━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ┫
┃               ┃   ┃   ┃           ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋   ╋   ┫
┃   ┃       ┃           ┃   ┃   ┃   ┃   ┃
┣   ╋   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━┫
┃       ┃   ┃       ┃       ┃   ┃       ┃
┣   ╋   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━┫
┃   ┃   ┃   ┃                   ┃   ┃   ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ┫
┃               ┃           ┃   ┃       ┃
┣   ╋   ╋━━━╋   ╋   ╋━━━╋━━━╋━━━╋   ╋   ┫
┃   ┃       ┃       ┃               ┃   ┃
┣   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━┫
┃   ┃           ┃       ┃       ┃   ┃   ┃
┣━━━╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋   ╋   ┫
┃           ┃   ┃   ┃           ┃   ┃   ┃
┣   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃       ┃               ┃       ┃       ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



---
## Section 3 - Résolution

### `solve_dfs`


In [11]:
laby = Maze.gen_fusion(10, 10)
solution = laby.solve_dfs((0, 0), (9, 9))
content = {c: '*' for c in solution}
content[(0, 0)] = 'D'
content[(9, 9)] = 'A'
print(laby.overlay(content))


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D   *   *         ┃   ┃   ┃       ┃   ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃         *   *   * ┃   ┃       ┃       ┃
┣━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋   ╋━━━┫
┃       ┃   ┃   ┃ *   *         ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ┫
┃                   ┃ * ┃               ┃
┣━━━╋━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋   ┫
┃       ┃   ┃         *   * ┃   ┃   ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━┫
┃   ┃           ┃   ┃   ┃ *   *         ┃
┣   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━┫
┃   ┃           ┃       ┃   ┃ *         ┃
┣   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃       ┃   ┃   ┃ *   * ┃       ┃
┣━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃       ┃           ┃   ┃ *   *   * ┃   ┃
┣━━━╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃           ┃                   ┃ *   A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `solve_bfs`

In [12]:
solution_bfs = laby.solve_bfs((0, 0), (9, 9))
content_bfs = {c: '*' for c in solution_bfs}
content_bfs[(0, 0)] = 'D'
content_bfs[(9, 9)] = 'A'
print(laby.overlay(content_bfs))


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D   *   *         ┃   ┃   ┃       ┃   ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃         *   *   * ┃   ┃       ┃       ┃
┣━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━╋   ╋━━━┫
┃       ┃   ┃   ┃ *   *         ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ┫
┃                   ┃ * ┃               ┃
┣━━━╋━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋   ┫
┃       ┃   ┃         *   * ┃   ┃   ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━┫
┃   ┃           ┃   ┃   ┃ *   *         ┃
┣   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━┫
┃   ┃           ┃       ┃   ┃ *         ┃
┣   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃       ┃   ┃   ┃ *   * ┃       ┃
┣━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃       ┃           ┃   ┃ *   *   * ┃   ┃
┣━━━╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃           ┃                   ┃ *   A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `solve_rhr` - main droite

In [13]:
"""solution_rhr = laby.solve_rhr((0, 0), (9, 9))
content_rhr = {c: '*' for c in solution_rhr}
content_rhr[(0, 0)] = 'D'
content_rhr[(9, 9)] = 'A'
print(laby.overlay(content_rhr))


TypeError: 'NoneType' object is not iterable

---
## Section 4 - Métriques

### Distances


In [19]:
laby_m = Maze.gen_fusion(10, 10)
c1, c2 = (0, 0), (9, 9)
print(laby_m)
print(f"Distance géodésique  {c1} → {c2} : {laby_m.distance_geo(c1, c2)}")
print(f"Distance de Manhattan {c1} → {c2} : {laby_m.distance_man(c1, c2)}")


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃       ┃               ┃           ┃   ┃
┣   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ┫
┃       ┃       ┃           ┃           ┃
┣   ╋   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋━━━┫
┃   ┃   ┃       ┃   ┃       ┃   ┃       ┃
┣   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ┫
┃   ┃   ┃   ┃           ┃           ┃   ┃
┣━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ┫
┃   ┃   ┃   ┃   ┃                       ┃
┣   ╋   ╋   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━┫
┃                           ┃       ┃   ┃
┣   ╋   ╋━━━╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ┫
┃   ┃   ┃               ┃       ┃       ┃
┣━━━╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━┫
┃           ┃           ┃               ┃
┣━━━╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋━━━┫
┃                       ┃               ┃
┣   ╋━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋   ╋━━━┫
┃   ┃           ┃   ┃       ┃   ┃       ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛

Distance géodésique  (0, 0) → (9, 9) : 20
Distance de Manhattan (0, 0) → (9, 9) : 18


### Tortuosité et longueur pire chemin

In [20]:
print(f"Tortuosité          : {laby_m.tortuosity(c1, c2):.3f}")
print(f"Longueur pire chemin: {laby_m.worst_path_len(c1, c2)}")


Tortuosité          : 1.111
Longueur pire chemin: 25


### Culs-de-sac et linéarité

In [21]:
print(f"Nombre de culs-de-sac : {laby_m.dead_end_number()}")
print(f"Taux de culs-de-sac   : {laby_m.dead_end_rate():.3f}")
print(f"Index de linéarité    : {laby_m.linearity_index():.3f}")


Nombre de culs-de-sac : None
Taux de culs-de-sac   : 0.340
Index de linéarité    : 0.410


---
## Section 5 - Labyrinthes imparfaits et valués

### `remove_random_wall`


In [ ]:
laby_imp = Maze.gen_fusion(8, 8)
print("Avant suppression d'un mur aléatoire :")
print(laby_imp.info())
laby_imp.remove_random_wall()
print("Après suppression d'un mur aléatoire :")
print(laby_imp.info())


### `add_random_weights` et degrés

In [ ]:
laby_w = Maze.gen_fusion(8, 8)
laby_w.add_random_weights(max_weight=10)
print(f"Degré moyen  : {laby_w.average_degree():.3f}")
print(f"Degré médian : {laby_w.median_degree():.3f}")
print(f"Extrait des poids : {dict(list(laby_w.weight.items())[:5])}")


---
## Section 6 - Résolution de labyrinthes valués

### `solve_dijkstra` (version naïve)


In [ ]:
laby_d = Maze.gen_fusion(10, 10)
laby_d.add_random_weights(max_weight=10)
start, stop = (0, 0), (9, 9)

path_dijk = laby_d.solve_dijkstra(start, stop)
content_d = {c: '*' for c in path_dijk}
content_d[start] = 'D'
content_d[stop]  = 'A'
print(laby_d.overlay(content_d))


### `solve_dijkstra_whq` (avec heapq)

In [ ]:
path_whq = laby_d.solve_dijkstra_whq(start, stop)
content_whq = {c: '*' for c in path_whq}
content_whq[start] = 'D'
content_whq[stop]  = 'A'
print(laby_d.overlay(content_whq))


### Comparaison des deux Dijkstra (temps d'exécution)

In [ ]:
import time

def time_solver(solver, laby, start, stop, n=50):
    t0 = time.perf_counter()
    for _ in range(n):
        solver(start, stop)
    return (time.perf_counter() - t0) / n * 1000  # ms

laby_bench = Maze.gen_fusion(20, 20)
laby_bench.add_random_weights(max_weight=10)
s, e = (0, 0), (19, 19)

t_naive = time_solver(laby_bench.solve_dijkstra,     laby_bench, s, e)
t_heapq = time_solver(laby_bench.solve_dijkstra_whq, laby_bench, s, e)

print(f"Dijkstra naïf  : {t_naive:.3f} ms")
print(f"Dijkstra heapq : {t_heapq:.3f} ms")
print(f"Accélération   : ×{t_naive/t_heapq:.1f}")


---
## Section 7 - Benchmark global des générateurs

Utiliser ce code une fois toutes les méthodes implémentées pour comparer les générateurs.


In [ ]:
def benchmark(generators, h, w, n):
    """Calcule les métriques moyennes pour chaque générateur."""
    results = {}
    for gen in generators:
        name = gen.__name__
        data = {k: [] for k in ['tortuosity','worst_path_len','dead_end_rate',
                                 'linearity_index','distance_geo','dead_end_number']}
        for _ in range(n):
            laby = gen(h, w)
            s, e = (0, 0), (h-1, w-1)
            data['tortuosity'].append(laby.tortuosity(s, e))
            data['worst_path_len'].append(laby.worst_path_len(s, e))
            data['dead_end_rate'].append(laby.dead_end_rate())
            data['linearity_index'].append(laby.linearity_index())
            data['distance_geo'].append(laby.distance_geo(s, e))
            data['dead_end_number'].append(laby.dead_end_number())
        results[name] = {k: sum(v)/n for k, v in data.items()}
    # Affichage simple en tableau
    header = f"{'Générateur':<20} " + " ".join(f"{k:>16}" for k in data)
    print(header)
    print("-" * len(header))
    for name, vals in results.items():
        row = f"{name:<20} " + " ".join(f"{v:>16.3f}" for v in vals.values())
        print(row)

generators = [Maze.gen_btree, Maze.gen_sidewinder, Maze.gen_fusion,
              Maze.gen_exploration, Maze.gen_wilson]
benchmark(generators, h=12, w=12, n=20)
